# Feature Enginnering
- Real-Worl data is not often neat and tidy and in addition to preprocessing steps like standardization we will likely have to extract and expand information from existing features . 
- Feature engineering is the creation of new features based on existing features .
- **Feature engineering** : Creation of new features from existing ones and it adds insight into relationships between features .
- Improve performance
- Insight into relationships between features
- Highly dataset-dependent


In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.float_format', '{:.2f}'.format)
hiking = pd.read_json("../7.Resources/hiking.json")
wine = pd.read_csv('../7.Resources/wine_types.csv')
volunteer = pd.read_csv("../7.Resources/volunteer_opportunities.csv") 

In [2]:
volunteer.head()

,opportunity_id,content_id,vol_requests,event_time,title,hits,summary,is_priority,category_id,category_desc,...,end_date_date,status,Latitude,Longitude,Community Board,Community Council,Census Tract,BIN,BBL,NTA
0,4996,37004,50,0,Volunteers Needed For Rise Up & Stay Put! Home...,737,Building on successful events last summer and ...,NaN,NaN,NaN,...,July 30 2011,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5008,37036,2,0,Web designer,22,Build a website for an Afghan business,NaN,1.00,Strengthening Communities,...,February 01 2011,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5016,37143,20,0,Urban Adventures - Ice Skating at Lasker Rink,62,Please join us and the students from Mott Hall...,NaN,1.00,Strengthening Communities,...,January 29 2011,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5022,37237,500,0,Fight global hunger and support women farmers ...,14,The Oxfam Action Corps is a group of dedicated...,NaN,1.00,Strengthening Communities,...,March 31 2012,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5055,37425,15,0,Stop 'N' Swap,31,Stop 'N' Swap reduces NYC's waste by finding n...,NaN,4.00,Environment,...,February 05 2011,approved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df = pd.DataFrame({
    'user_id': [101, 102, 103, 104, 105, 106, 107],
    'age': [19, 22, 25, 21, 24, 23, 20],
    'subscribed': ['yes', 'no', 'yes', 'no', 'yes', 'yes', 'no'],  # Binary categorical
    'favorite_color': ['red', 'blue', 'green', 'blue', 'red', 'yellow', 'green']  # Multi-category
})

print(df)

   user_id  age subscribed favorite_color
0      101   19        yes            red
1      102   22         no           blue
2      103   25        yes          green
3      104   21         no           blue
4      105   24        yes            red
5      106   23        yes         yellow
6      107   20         no          green


# Encoding binary Variables - pandas

In [4]:
print(df['subscribed'])

0    yes
1     no
2    yes
3     no
4    yes
5    yes
6     no
Name: subscribed, dtype: str


In [5]:
df['sub_enc'] = df['subscribed'].apply(lambda val : 1 if val == "yes" else 0)

In [6]:
print(df[["subscribed","sub_enc"]])

  subscribed  sub_enc
0        yes        1
1         no        0
2        yes        1
3         no        0
4        yes        1
5        yes        1
6         no        0


# Encoding binary variables - scikit-learn 

In [7]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df['sub_enc_le'] = le.fit_transform(df['subscribed'])

print(df[['subscribed','sub_enc_le']])

  subscribed  sub_enc_le
0        yes           1
1         no           0
2        yes           1
3         no           0
4        yes           1
5        yes           1
6         no           0


# One hot encoding

In [8]:
print(df['favorite_color'])

0       red
1      blue
2     green
3      blue
4       red
5    yellow
6     green
Name: favorite_color, dtype: str


In [9]:
print(pd.get_dummies(df['favorite_color']))

    blue  green    red  yellow
0  False  False   True   False
1   True  False  False   False
2  False   True  False   False
3   True  False  False   False
4  False  False   True   False
5  False  False  False    True
6  False   True  False   False


# Engineerig Numerical Features

In [10]:
data = {
    'City': ['Karachi', 'Lahore', 'Islamabad', 'Quetta'],
    'Day_1': [30, 28, 25, 20],
    'Day_2': [31, 29, 26, 21],
    'Day_3': [29, 30, 24, 19],
    'Day_4': [32, 31, 27, 22],
    'Day_5': [30, 29, 25, 18]
}
temps = pd.DataFrame(data)


In [11]:
temps['mean'] = temps.loc[:,'Day_1':'Day_2'].mean(axis=1)
print(temps)

        City  Day_1  Day_2  Day_3  Day_4  Day_5  mean
0    Karachi     30     31     29     32     30 30.50
1     Lahore     28     29     30     31     29 28.50
2  Islamabad     25     26     24     27     25 25.50
3     Quetta     20     21     19     22     18 20.50


In [12]:
data = {
    "Purchase_Date": [
        "05 January 2026",
        "12 January 2026",
        "03 February 2026",
        "15 February 2026",
        "01 March 2026"
    ],
    "Purchase_Price": [2500, 1800, 3200, 1500, 2700]
}

purchases = pd.DataFrame(data)

purchases['date_converted'] = pd.to_datetime(purchases['Purchase_Date'])
purchases['month'] = purchases['date_converted'].dt.month
purchases['year'] = purchases['date_converted'].dt.year

print(purchases)

      Purchase_Date  Purchase_Price date_converted  month  year
0   05 January 2026            2500     2026-01-05      1  2026
1   12 January 2026            1800     2026-01-12      1  2026
2  03 February 2026            3200     2026-02-03      2  2026
3  15 February 2026            1500     2026-02-15      2  2026
4     01 March 2026            2700     2026-03-01      3  2026


# Engineering Text Features

- One method to extract the pieces of information that you need : maybe part of a string or extracting numbers and transforming it 
- We are going to use regular expressions to extract information from text datta that can be used to extract information from text data

- **Regular expressions** : code to identify patterns

In [13]:
import re
my_string = "temperature:75.6 F"
temp = re.search(r"\d+\.\d+",my_string)
print(float(temp.group(0)))

75.6


# Vectorizing Text
- **TF/IDF** : Vectorizes words based upon importance
- TF = Term Frequency
- IDF = Inverse Document Frequency

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = hiking['Location']

tfidf_vec = TfidfVectorizer()
text_tfidf = tfidf_vec.fit_transform(documents)

print(tfidf_vec.vocabulary_)

{'enter': 31, 'behind': 14, 'the': 102, 'salt': 93, 'marsh': 63, 'nature': 69, 'center': 20, 'located': 59, 'near': 70, 'intersection': 52, 'of': 73, 'east': 29, '33rd': 2, 'street': 100, 'and': 5, 'avenue': 10, 'park': 80, 'at': 8, 'lincoln': 58, 'road': 90, 'ocean': 72, 'entrance': 32, 'trails': 106, 'begin': 13, 'or': 76, 'are': 7, 'prospect': 86, 'audubon': 9, 'wide': 116, 'check': 21, 'out': 78, 'our': 77, 'href': 48, 'features': 35, 'hiking': 44, 'alley': 4, 'pond': 85, 'page': 79, 'for': 38, 'map': 62, 'directions': 27, 'to': 103, 'scenic': 94, 'locations': 60, 'forest': 39, 'drive': 28, 'off': 74, 'woodhaven': 118, 'boulevard': 16, 'memorial': 64, 'metropolitan': 65, 'francis': 40, 'lewis': 56, 'union': 109, 'turnpike': 108, 'brielle': 17, 'roanoake': 91, 'rockland': 92, 'avenues': 11, 'richmond': 89, 'st': 97, 'patrick': 83, 'place': 84, 'mid': 66, 'way': 114, 'trailhead': 105, 'reidel': 88, 'hill': 45, 'mulberry': 68, 'travis': 107, 'limited': 57, 'parking': 81, 'willowbrook'